# DLT Prototype V2

Let's do some performance benchmarking comparing to Orbit

## Library and Settings

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import logging

# import os
# os.environ["XLA_FLAGS"] = "--xla_cpu_multi_thread_eigen=true --xla_cpu_multi_thread_eigen_num_threads=4"
import numpyro
numpyro.set_host_device_count(4)
numpyro.enable_x64(True)
import matplotlib.pyplot as plt

from wunku.models.dlt import (
    run_dlt_model, 
    generate_in_sample_forecast, 
)

from wunku.utils import (
    summarize_posteriors,
    make_fourier_series,
)

In [ ]:
# Get or create logger
logger = logging.getLogger("wunku")

# Set logger level
logger.setLevel(logging.DEBUG)

# Add a StreamHandler if no handlers are attached
if not logger.hasHandlers():
    handler = logging.StreamHandler()
    handler.setLevel(logging.DEBUG)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)

    logger.addHandler(handler)

# Example usage
logger.debug("This is a DEBUG message")
logger.info("This is an INFO message")

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# model_tag = "missing-values-seasonal-events"
model_tag = "seasonal-events"

## Data

In [ ]:
df = pd.read_csv(f"../resource/{model_tag}/df.csv", parse_dates=["date"])
train_df = df[:-100]
test_df = df[-100:]
saturation_df = pd.read_csv(f"../resource/{model_tag}/saturation_df.csv").set_index("regressor")
response = train_df["sales"].values
# response_norm = response.mean()
response_norm = 1.
y = np.log(response / response_norm)

resource_cols = [
    "promo", "radio", "search", "social", "tv"
]

# we should encode a miss-value signal
train_df[resource_cols] = train_df[resource_cols].fillna(0)

event_cols = [
    "new-years-day",
    "martin-luther-king-jr-day",
    "washingtons-birthday",
    "memorial-day",
    "independence-day",
    "labor-day",
    "columbus-day",
    "veterans-day",
    "thanksgiving-day",
    "christmas-day",
    "independence-day-observed",
    "juneteenth-national-independence-day-observed",
    "juneteenth-national-independence-day",
    "christmas-day-observed",
    "new-years-day-observed",
]
# ['independence-day', 'juneteenth-national-independence-day', 'labor-day', 'memorial-day', 'new-years-day', 'new-years-day-observed', 'thanksgiving-day', 'veterans-day', 'veterans-day-observed', 'washingtons-birthday']
event_cols = [x for x in event_cols if x in df.columns]

## Train model and data process

In [ ]:
covariates = []
coef_loc = []
coef_scale = []
var_names = []

x_glb_intercept = np.ones((len(y), 1))
x_glb_slp = np.expand_dims(np.arange(len(y)) / 365.25, -1)
covariates += [x_glb_intercept, x_glb_slp]

coef_loc_glb_intercept = 0
coef_loc_glb_slp = 0
coef_scale_glb_intercept = 1.
coef_scale_glb_slp = 0.001

coef_loc += [coef_loc_glb_intercept, coef_loc_glb_slp]
coef_scale += [coef_scale_glb_intercept, coef_scale_glb_slp]
var_names += ["glb_intercept", "glb_slp",]

if model_tag in ["seasonal-events", "seasonal-event-adstock", "missing-values-seasonal-events"]:
    x_annual_seas = make_fourier_series(np.arange(len(y)), period=365.25, order=3)
    x_weekly_seas = make_fourier_series(np.arange(len(y)), period=7, order=2)
    x_seas = np.concatenate((x_annual_seas, x_weekly_seas), axis=1)
    covariates.append(x_seas)
    coef_loc_seas = [0.] * x_seas.shape[1]
    coef_scale_seas = [0.3] * x_seas.shape[1]

    coef_loc += coef_loc_seas
    coef_scale += coef_scale_seas

    var_names += [f"seas_{i}" for i in range(x_seas.shape[1])]

if model_tag in ["seasonal-events", "events", "missing-values-seasonal-events"]:
    x_event = train_df[event_cols].values
    covariates.append(x_event)
    coef_loc_event = [0.] * x_event.shape[1]
    coef_scale_event = [1.0] * x_event.shape[1]
    coef_loc += coef_loc_event
    coef_scale += coef_scale_event
    var_names += event_cols

x_resource = train_df[resource_cols].values
sat_arr = saturation_df.loc[resource_cols, "saturation"].values
x_resource = np.log1p(x_resource / sat_arr)
covariates.append(x_resource)
coef_loc_resource = [0] * x_resource.shape[1]
coef_scale_resource = [0.1] * x_resource.shape[1]
coef_loc += coef_loc_resource
coef_scale += coef_scale_resource
var_names += resource_cols

# stack all covariates and related input
covariates = np.concatenate(covariates, axis=1)
coef_loc = np.array(coef_loc)
coef_scale = np.array(coef_scale)
var_name = np.array(var_names)

print(f"var_names: {var_name}")
print(f"coef_loc shape: {coef_loc.shape}")
print(f"coef_scale shape: {coef_scale.shape}")
print(f"covariates shape: {covariates.shape}")

In [ ]:
# from our best model last time
lev_sm=0.0266
# lev_sm = 0.001
slp_sm=0.001
# theta=0.94900
theta=0.00

In [ ]:
data_vars = {
    "covariates": (["time", "var_name"], covariates),
    "coef_loc": (["var_name"], coef_loc),
    "coef_scale": (["var_name"], coef_scale),
}
coords = {
    "time": np.arange(len(y)),
    "var_name": var_name,
}
regression_scheme = xr.Dataset(
    data_vars=data_vars,
    coords=coords,
)
regression_scheme

In [ ]:
%%time
posteriors = run_dlt_model(
    y=y,
    lev_sm=lev_sm,
    slp_sm=slp_sm,
    theta=theta,
    regression_scheme=regression_scheme,
    mcmc_run_args={
        "num_warmup": 250,
        "num_samples": 250,
        "num_chains": 4,
        "progress_bar": True,
    },
)

## Diagnostics

In [ ]:
import arviz as az
summary_df = az.summary(
    posteriors,
    # coords=("var_name"),
    var_names=["coef", "sigma"],
    round_to=5,
)
summary_df

In [ ]:
dlt_comp = posteriors["dlt_comp"].to_numpy()
dlt_comp = dlt_comp.reshape(-1, *dlt_comp.shape[2:])
dlt_comp = np.mean(dlt_comp, axis=0)
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
ax.plot(dlt_comp, label="level")

In [ ]:
coefs_df = pd.read_csv(f"../resource/{model_tag}/coefs_df.csv", index_col="regressor")
coefs_df

In [ ]:
coef_names = [f"coef[{x}]" for x in resource_cols]
coef_p50 = summary_df.loc[coef_names, "mean"].values
coef_true = coefs_df.loc[resource_cols, "coef"].values
print(f"Coef P50: {coef_p50}")
print(f"Coef True: {coef_true}")
mse_err = np.mean(
    np.abs(coef_true - coef_p50)
)

smape_err = np.mean(
    2 * np.abs(coef_true - coef_p50) / (np.abs(coef_true) + np.abs(coef_p50))
)

print(f"Coef SMAPE (%): {smape_err:.3%}")
print(f"MSE: {mse_err:.4f}")

## Forecast

### out-of-sample DLT

In [ ]:
sigma = posteriors["sigma"].to_numpy()
last_lev = posteriors["last_lev"].to_numpy()
last_slp = posteriors["last_slp"].to_numpy()
coef = posteriors["coef"].to_numpy()

# flatten first 2 dimensions
sigma = sigma.reshape(-1, *sigma.shape[2:])
coef = coef.reshape(-1, *coef.shape[2:])
last_lev = last_lev.reshape(-1, *last_lev.shape[2:])
last_slp = last_slp.reshape(-1, *last_slp.shape[2:])

# report shapes
print(f"sigma shape: {sigma.shape}")
print(f"coef shape: {coef.shape}")
print(f"last_lev shape: {last_lev.shape}")
print(f"last_slp shape: {last_slp.shape}")

In [ ]:
import jax
from jax import lax
n_steps = 100
rngkey = jax.random.PRNGKey(2025)
# (n_steps, n_samples)
eps = jax.random.normal(rngkey, shape=(n_steps, sigma.shape[0]))
print(f"eps shape: {eps.shape}")  

def dlt_forecast_transition_step(carry, inputs, lev_sm, slp_sm, theta):
    lev_prev, slp_prev = carry
    eps_t = inputs

    # forecast
    dlt_comp_t = lev_prev + theta * slp_prev
    # simulated obs
    y_t = dlt_comp_t + eps_t

    # update level and slope
    new_lev = lev_sm * y_t + (1 - lev_sm) * (dlt_comp_t)
    new_slp = slp_sm * (new_lev - lev_prev) + (1 - slp_sm) * slp_prev

    return (new_lev, new_slp), (new_lev, new_slp, dlt_comp_t)

# scan with the partial function
_, all_states = lax.scan(
    lambda carry, inputs: dlt_forecast_transition_step(carry, inputs, lev_sm, slp_sm, theta), 
    (last_lev, last_slp), eps,
)
_, _, dlt_comp_forecast = all_states

### Regression (Forecast)

In [ ]:
covariates = []
n_steps = test_df.shape[0]
n_train_steps = train_df.shape[0]
x_glb_intercept = np.ones((n_steps, 1))
x_glb_slp = np.expand_dims(np.arange(n_train_steps, n_train_steps + n_steps) / 365.25, -1)
covariates += [x_glb_intercept, x_glb_slp]

if model_tag in ["seasonal-events", "seasonal-event-adstock", "missing-values-seasonal-events"]:
    x_annual_seas = make_fourier_series(np.arange(n_train_steps, n_train_steps + n_steps), period=365.25, order=3)
    x_weekly_seas = make_fourier_series(np.arange(n_train_steps, n_train_steps + n_steps), period=7, order=2)
    x_seas = np.concatenate((x_annual_seas, x_weekly_seas), axis=1)
    covariates.append(x_seas)

if model_tag in ["seasonal-events", "events", "missing-values-seasonal-events"]:
    x_event = test_df[event_cols].values
    covariates.append(x_event)

x_resource = test_df[resource_cols].values
sat_arr = saturation_df.loc[resource_cols, "saturation"].values
x_resource = np.log1p(x_resource / sat_arr)
covariates.append(x_resource)

# stack all covariates and related input
covariates = np.concatenate(covariates, axis=1)
print(f"covariates shape: {covariates.shape}")


In [ ]:
yhat_is_lower, yhat_is_mid, yhat_is_upper = generate_in_sample_forecast(
    posteriors=posteriors,
    q=0.05,
)

In [ ]:
dlt_comp_forecast_samples = dlt_comp_forecast.transpose(-1, -2)
# (n_samples, n_var) * (n_steps, n_var) -> (n_samples, n_steps)
reg_comp_samples = np.einsum("ik,jk->ij", coef, covariates)
print(f"reg_comp_samples shape: {reg_comp_samples.shape}")

yhat_oos_lower, yhat_oos_mid, yhat_oos_upper = np.percentile(
    np.exp(reg_comp_samples + dlt_comp_forecast_samples), 
    q=[2.5, 50, 97.5], 
    axis=0
)

In [ ]:
yhat_lower = np.concatenate((yhat_is_lower, yhat_oos_lower), axis=0)
yhat_mid = np.concatenate((yhat_is_mid, yhat_oos_mid), axis=0)
yhat_upper = np.concatenate((yhat_is_upper, yhat_oos_upper), axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# in-sample data plot
ax.scatter(df["date"].values, df["sales"].values, label='Observed', alpha=0.5, color="orange", s=10)
ax.plot(df["date"].values, yhat_mid, label='Forecast', linestyle='--', alpha=0.8, color="dodgerblue")
ax.fill_between(df["date"].values, yhat_lower, yhat_upper, alpha=0.5, label='95% Prediction Interval', color="dodgerblue")

# vertical line indicate the forecast start
ax.axvline(test_df["date"].values[0], color='gray', linestyle='--', label='Forecast Start')

ax.set_title('Damped Local Trend Forecast')
